## CMA-ES 协方差矩阵自适应进化策略

### CMA-ES 的起源

CMA-ES（Covariance Matrix Adaptation Evolution Strategy）由 **Nikolaus Hansen** 和 **Andreas Ostermeier** 于 1996 年提出，是进化策略（Evolution Strategy, ES）家族中最成功的变体之一。

**进化策略的演化脉络：**
- **(1+1)-ES**（Rechenberg, 1965）：最简单的 ES，每次只产生一个后代
- **($\mu$/$\mu_w$, $\lambda$)-ES**（Schwefel, 1970s）：引入种群和重组
- **CMA-ES**（Hansen & Ostermeier, 1996）：自适应学习协方差矩阵，显著提升在 ill-conditioned 和非凸问题上的表现

### 核心思想

传统进化策略使用固定方差或各向同性的高斯分布进行搜索。CMA-ES 的核心创新在于：

1. **自适应协方差矩阵**：通过迭代学习搜索分布的二阶统计量，使采样分布逐步"贴合"目标函数的局部几何结构
2. **步长控制**：通过累积路径长度（Cumulative Step-size Adaptation, CSA）独立调整全局步长
3. **不变性**：CMA-ES 对平移、旋转和尺度变换具有不变性，这是其性能优异的关键

### CMA-ES 与 DE 的对比

| 特性 | DE | CMA-ES |
|------|-----|--------|
| 搜索分布 | 隐式（差分向量） | 显式（多元正态分布） |
| 参数自适应 | 无（固定 $F$, $CR$） | 协方差矩阵 + 步长全自适应 |
| 种群规模 | 较小（$NP \approx 5D$~$10D$） | 较小（$\lambda \approx 4 + 3\ln D$） |
| 高维性能 | 一般 | 优秀（20~100 维） |
| 不变性 | 旋转不变 | 平移/旋转/尺度均不变 |

**优点：** 参数自适应、不变性好、无需手动调参、在 ill-conditioned 问题上表现优异
**缺点：** 时间复杂度 $O(D^2)$（受协方差矩阵分解影响）、实现复杂度较高

## CMA-ES 的基本原理

### 1. 采样（Sampling）

在第 $g$ 代，从多元正态分布中采样 $\lambda$ 个候选解：

$$
\mathbf{x}_k^{(g+1)} = \mathbf{m}^{(g)} + \sigma^{(g)} \mathbf{B}^{(g)} \mathbf{D}^{(g)} \mathbf{z}_k,
\quad \mathbf{z}_k \sim \mathcal{N}(0, \mathbf{I}), \quad k = 1, 2, \dots, \lambda
$$

其中协方差矩阵 $\mathbf{C}^{(g)} = \mathbf{B}^{(g)} (\mathbf{D}^{(g)})^2 (\mathbf{B}^{(g)})^T$，$\mathbf{B}$ 为正交特征向量矩阵，$\mathbf{D}$ 为特征值平方根的对角阵。

### 2. 选择与重组（Selection & Recombination）

将 $\lambda$ 个后代按适应度排序（最小化问题则升序），取前 $\mu$ 个个体进行加权重组：

$$
\mathbf{m}^{(g+1)} = \mathbf{m}^{(g)} + c_m \sum_{i=1}^{\mu} w_i \left( \mathbf{x}_{i:\lambda}^{(g+1)} - \mathbf{m}^{(g)} \right)
$$

其中 $w_i$ 为权重，满足 $\sum_{i=1}^{\mu} w_i = 1$，$w_1 \geq w_2 \geq \dots \geq w_\mu > 0$（通常 $c_m = 1$）。

### 3. 协方差矩阵更新（Covariance Matrix Update）

协方差矩阵 $\mathbf{C}$ 通过两种机制更新：

**Rank-$\mu$ 更新：** 利用当前种群中 $\mu$ 个最优个体的分布信息

$$
\mathbf{C}_\mu^{(g+1)} = \sum_{i=1}^{\mu} w_i \left( \frac{\mathbf{x}_{i:\lambda}^{(g+1)} - \mathbf{m}^{(g)}}{\sigma^{(g)}} \right) \left( \frac{\mathbf{x}_{i:\lambda}^{(g+1)} - \mathbf{m}^{(g)}}{\sigma^{(g)}} \right)^T
$$

**Rank-1 更新：** 利用演化路径 $\mathbf{p}_c$ 捕获代际相关性

$$
\mathbf{C}_1^{(g+1)} = \mathbf{p}_c^{(g+1)} (\mathbf{p}_c^{(g+1)})^T
$$

综合更新公式：

$$
\mathbf{C}^{(g+1)} = (1 - c_1 - c_\mu) \mathbf{C}^{(g)} + c_1 \mathbf{C}_1^{(g+1)} + c_\mu \mathbf{C}_\mu^{(g+1)}
$$

### 4. 步长控制（Step-size Control, CSA）

通过共轭演化路径 $\mathbf{p}_\sigma$ 控制全局步长 $\sigma$：

$$
\mathbf{p}_\sigma^{(g+1)} = (1 - c_\sigma) \mathbf{p}_\sigma^{(g)} + \sqrt{c_\sigma (2 - c_\sigma) \mu_{\mathrm{eff}}} \; \mathbf{C}^{(g)-1/2} \; \frac{\mathbf{m}^{(g+1)} - \mathbf{m}^{(g)}}{\sigma^{(g)}}
$$

$$
\sigma^{(g+1)} = \sigma^{(g)} \cdot \exp\left( \frac{c_\sigma}{d_\sigma} \left( \frac{\| \mathbf{p}_\sigma^{(g+1)} \|}{\mathbb{E}\|\mathcal{N}(0, \mathbf{I})\|} - 1 \right) \right)
$$

- 当 $\|\mathbf{p}_\sigma\| > \mathbb{E}\|\mathcal{N}(0, \mathbf{I})\|$ 时：增大 $\sigma$，加强探索
- 当 $\|\mathbf{p}_\sigma\| < \mathbb{E}\|\mathcal{N}(0, \mathbf{I})\|$ 时：减小 $\sigma$，加强开发

其中 $\mathbb{E}\|\mathcal{N}(0, \mathbf{I})\| \approx \sqrt{N}\left(1 - \frac{1}{4N} + \frac{1}{21N^2}\right)$ 为标准正态分布范数的期望值。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from mpl_toolkits.mplot3d import Axes3D

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

## 实例计算

本案例使用 **Rastrigin 函数** 作为优化目标，该函数具有大量局部极小值，是测试全局优化算法的经典基准函数：

$$
f(\mathbf{x}) = 10D + \sum_{i=1}^{D} \left[ x_i^2 - 10 \cos(2\pi x_i) \right]
$$

其中 $D$ 为维度，搜索范围 $x_i \in [-5.12, 5.12]$。全局最小值在 $\mathbf{x} = (0, 0, \dots, 0)$ 处，$f_{\min} = 0$。

In [ ]:
# 定义 Rastrigin 函数
def rastrigin(x):
    '''Rastrigin 函数，x 形状为 (N, D)'''
    D = x.shape[1]
    return 10 * D + np.sum(x**2 - 10 * np.cos(2 * np.pi * x), axis=1)

# 绘制二维 Rastrigin 函数
x_vals = np.linspace(-5.12, 5.12, 100)
y_vals = np.linspace(-5.12, 5.12, 100)
X, Y = np.meshgrid(x_vals, y_vals)
points = np.column_stack([X.ravel(), Y.ravel()])
Z = rastrigin(points).reshape(X.shape)

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)
ax.contour(X, Y, Z, zdir='z', offset=Z.min(), cmap='viridis', alpha=0.5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('f(x, y)')
ax.set_title('Rastrigin 函数图像')
fig.colorbar(surf, shrink=0.5, aspect=10)
plt.show()

### CMA-ES 参数设置

CMA-ES 的一个显著优势是大部分策略参数可以**自动推导**，用户只需设置：

- **$D$**：问题维度
- **$\sigma_0$**：初始步长
- **$\mathbf{m}_0$**：初始均值（通常随机初始化）

其他参数按 Hansen (2016) 的默认公式自动计算。

---

#### 各学习率的推导思路（精简版）

这些公式**不是纯经验的，但也不全是严格解析推导**，而是来自**理论约束下的渐进分析 + 实验拟合**：

**$c_\sigma$（步长路径学习率）** — 理论为主（~80%）

基于平稳分布约束：$\mathbf{p}_\sigma$ 在随机选择下应服从 $\mathcal{N}(0,\mathbf{I})$。$c_\sigma$ 决定路径的记忆长度，应为 $1/\mu_{\mathrm{eff}}$ 量级；分母中出现 $D$ 是因为高维下每代信息量下降。$+2$ 和 $+5$ 是低 $\mu_{\mathrm{eff}}$ 时防学习率过小的经验常数。

**$d_\sigma$（步长阻尼）** — 理论与经验各半

控制 $\sigma$ 的变化速率。$\mu_{\mathrm{eff}}$ 大时样本方差低、可加速调整；$\mu_{\mathrm{eff}}$ 接近 $1$ 时需阻尼防震荡。$\max(0, \dots)$ 中的具体常数来自实验调优。

**$c_c$（协方差路径学习率）** — 中偏理论（~60%）

同基于平稳分布约束。渐近形式 $c_c \approx 4/D$：协方差矩阵有 $D^2$ 个参数，每代仅 $\mu_{\mathrm{eff}}$ 个有效样本，故学习率正比于 $\mu_{\mathrm{eff}}/D^2$。常数 $4$ 和 $2$ 来自实验调优。

**$c_1$ 和 $c_\mu$（协方差更新学习率）** — 经验为主（~40% 理论）

- $c_1 \approx 2/D^2$：rank-1 更新每次贡献 $1$ 个自由度，学习率反比于协方差矩阵的 $D^2$ 个自由度
- $c_\mu \approx \mu_{\mathrm{eff}}/D^2$：rank-$\mu$ 有 $\mu$ 个有效样本，信息量更大，学习率应更大
- $(D+1.3)^2$ 中的 $1.3$、$(D+2)^2$ 中的 $2$ 均为实验常数
- $\min(1-c_1,\dots)$ 是硬约束：确保 $1-c_1-c_\mu > 0$

> 这些默认值是 Hansen 在几十个测试函数上经大量实验验证的推荐值，适用于绝大多数场景。

---

#### 具体公式

- $\lambda = 4 + \lfloor 3\ln D \rfloor$（种群规模）
- $\mu = \lfloor \lambda/2 \rfloor$（父代数量）
- $w_i = \dfrac{\ln\frac{\lambda+1}{2} - \ln i}{\sum_{j=1}^{\mu}\left(\ln\frac{\lambda+1}{2} - \ln j\right)}$（重组权重）
- $\mu_{\mathrm{eff}} = \left(\sum w_i^2\right)^{-1}$（有效方差选择质量）
- $c_\sigma = \dfrac{\mu_{\mathrm{eff}} + 2}{D + \mu_{\mathrm{eff}} + 5}$
- $d_\sigma = 1 + 2\max\!\left(0,\sqrt{\frac{\mu_{\mathrm{eff}}-1}{D+1}}-1\right) + c_\sigma$
- $c_c = \dfrac{4 + \mu_{\mathrm{eff}}/D}{D + 4 + 2\mu_{\mathrm{eff}}/D}$
- $c_1 = \dfrac{2}{(D+1.3)^2 + \mu_{\mathrm{eff}}}$
- $c_\mu = \min\!\left(1-c_1,\; \dfrac{2(\mu_{\mathrm{eff}}-2+1/\mu_{\mathrm{eff}})}{(D+2)^2 + \mu_{\mathrm{eff}}}\right)$

In [ ]:
# ==================== CMA-ES 参数设置 ====================
D = 10                 # 问题维度
sigma0 = 0.5           # 初始步长
max_gen = 500          # 最大迭代次数

# 搜索范围
x_min, x_max = -5.12, 5.12

# ---------- 自动推导的策略参数（Hansen, 2016）----------
lam = 4 + int(np.floor(3 * np.log(D)))   # 种群规模 lambda
mu = lam // 2                              # 父代数量 mu

# 重组权重
weights = np.log((lam + 1) / 2) - np.log(np.arange(1, mu + 1))
weights = weights / np.sum(weights)        # 归一化
mu_eff = 1.0 / np.sum(weights**2)          # 有效方差选择质量

# 学习率
c_sigma = (mu_eff + 2) / (D + mu_eff + 5)
d_sigma = 1 + 2 * max(0, np.sqrt((mu_eff - 1) / (D + 1)) - 1) + c_sigma
c_c = (4 + mu_eff / D) / (D + 4 + 2 * mu_eff / D)
c_1 = 2 / ((D + 1.3)**2 + mu_eff)
c_mu = min(1 - c_1, 2 * (mu_eff - 2 + 1 / mu_eff) / ((D + 2)**2 + mu_eff))

# 期望值常数
chi_N = np.sqrt(D) * (1 - 1 / (4 * D) + 1 / (21 * D**2))

print(f'问题维度 D = {D}')
print(f'初始步长 sigma0 = {sigma0}')
print(f'种群规模 lambda = {lam}')
print(f'父代数量 mu = {mu}')
print(f'有效方差选择质量 mu_eff = {mu_eff:.3f}')
print(f'最大迭代次数 = {max_gen}')
print(f'搜索范围: [{x_min}, {x_max}]')
print()
print('--- 自动推导的学习率 ---')
print(f'c_sigma = {c_sigma:.4f}')
print(f'd_sigma = {d_sigma:.4f}')
print(f'c_c = {c_c:.4f}')
print(f'c_1   = {c_1:.4f}')
print(f'c_mu  = {c_mu:.4f}')

### 采样操作

从当前搜索分布 $\mathcal{N}(\mathbf{m}, \sigma^2 \mathbf{C})$ 中采样 $\lambda$ 个后代。先对协方差矩阵做特征分解 $\mathbf{C} = \mathbf{B} \mathbf{D}^2 \mathbf{B}^T$，然后：

$$
\mathbf{x}_k = \mathbf{m} + \sigma \cdot \mathbf{B} \mathbf{D} \cdot \mathbf{z}_k, \quad \mathbf{z}_k \sim \mathcal{N}(0, \mathbf{I})
$$

其中 $\mathbf{B}$ 的列是 $\mathbf{C}$ 的标准正交特征向量，$\mathbf{D}$ 是特征值平方根构成的对角矩阵。

In [ ]:
def eigen_decompose(C):
    '''
    对协方差矩阵进行特征分解 C = B D^2 B^T
    返回 B, D (D 是标准差，即特征值的平方根)
    '''
    eigenvalues, eigenvectors = np.linalg.eigh(C)
    # 确保正定：将过小的特征值限制为一个很小的正值
    eigenvalues = np.maximum(eigenvalues, 1e-30)
    D = np.sqrt(eigenvalues)
    B = eigenvectors
    return B, D

def sample_population(m, sigma, B, D, lam):
    '''
    从 N(m, sigma^2 * C) 采样 lam 个后代
    其中 C = B @ diag(D^2) @ B^T
    '''
    D_dim = len(m)
    z = np.random.randn(lam, D_dim)                # 标准正态
    y = z @ np.diag(D) @ B.T                        # B D z
    pop = m + sigma * y                              # m + sigma * B D z
    return pop

### 选择与重组

将采样得到的 $\lambda$ 个个体按适应度从小到大排序（最小化问题），取前 $\mu$ 个进行加权平均来更新均值：

$$
\mathbf{m}_{\mathrm{new}} = \sum_{i=1}^{\mu} w_i \cdot \mathbf{x}_{i:\lambda}
$$

注意这里用 $\mu$ 个最优个体的加权平均直接替换均值（即 $c_m = 1$）。

In [ ]:
def select_and_recombine(pop, fitness, weights, mu):
    '''
    选择前 mu 个最优个体，加权重组得到新均值
    返回: m_new, selected (选中的 mu 个个体)
    '''
    # 按适应度升序排序
    idx = np.argsort(fitness)
    selected = pop[idx[:mu]]

    # 加权平均
    m_new = np.sum(selected * weights[:, np.newaxis], axis=0)

    return m_new, selected

### 协方差矩阵更新

协方差矩阵的更新由两部分组成：

1. **Rank-$\mu$ 更新**：利用 $\mu$ 个最优个体的样本协方差
2. **Rank-1 更新**：利用演化路径 $\mathbf{p}_c$ 的秩-1 外积

演化路径 $\mathbf{p}_c$ 累积了均值的移动方向，捕获代际相关性：

$$
\mathbf{p}_c^{(g+1)} = (1 - c_c) \mathbf{p}_c^{(g)} + \sqrt{c_c(2 - c_c)\mu_{\mathrm{eff}}} \cdot \frac{\mathbf{m}^{(g+1)} - \mathbf{m}^{(g)}}{\sigma^{(g)}}
$$

协方差更新：

$$
\mathbf{C}^{(g+1)} = (1 - c_1 - c_\mu) \mathbf{C}^{(g)} + c_1 \underbrace{\mathbf{p}_c \mathbf{p}_c^T}_{\mathrm{rank\text{-}1}} + c_\mu \underbrace{\sum_{i=1}^{\mu} w_i \mathbf{y}_{i:\lambda} \mathbf{y}_{i:\lambda}^T}_{\mathrm{rank\text{-}}\mu}
$$

其中 $\mathbf{y}_{i:\lambda} = (\mathbf{x}_{i:\lambda} - \mathbf{m}^{(g)}) / \sigma^{(g)}$。

In [ ]:
def update_covariance(C, pc, y_weighted, selected, m_old, m_new, sigma, weights, c_1, c_mu, c_c, mu_eff):
    '''
    更新协方差矩阵 C 和演化路径 pc
    y_weighted: 加权平均的 (m_new - m_old) / sigma，用于 pc 更新
    '''
    D = len(m_old)
    mu = len(selected)

    # 更新演化路径 p_c
    pc = (1 - c_c) * pc + np.sqrt(c_c * (2 - c_c) * mu_eff) * y_weighted

    # y_i = (x_{i:lambda} - m_old) / sigma
    y = (selected - m_old) / sigma

    # Rank-mu 更新
    C_mu = np.zeros((D, D))
    for i in range(mu):
        C_mu += weights[i] * np.outer(y[i], y[i])

    # 综合更新
    C = (1 - c_1 - c_mu) * C + c_1 * np.outer(pc, pc) + c_mu * C_mu

    return C, pc

### 步长 $\sigma$ 更新（CSA 机制）

通过共轭演化路径 $\mathbf{p}_\sigma$ 的长度与期望长度的比值来调整步长：

$$
\mathbf{p}_\sigma^{(g+1)} = (1 - c_\sigma) \mathbf{p}_\sigma^{(g)} + \sqrt{c_\sigma (2 - c_\sigma) \mu_{\mathrm{eff}}} \; \mathbf{C}^{(g)-1/2} \; \frac{\mathbf{m}^{(g+1)} - \mathbf{m}^{(g)}}{\sigma^{(g)}}
$$

$$
\sigma^{(g+1)} = \sigma^{(g)} \cdot \exp\left( \frac{c_\sigma}{d_\sigma} \left( \frac{\| \mathbf{p}_\sigma^{(g+1)} \|}{\chi_N} - 1 \right) \right)
$$

- 当 $\|\mathbf{p}_\sigma\| > \chi_N$（路径比期望长）：增大 $\sigma$，加强探索
- 当 $\|\mathbf{p}_\sigma\| < \chi_N$（路径比期望短）：减小 $\sigma$，加强开发

其中 $\chi_N = \mathbb{E}\|\mathcal{N}(0, \mathbf{I})\| \approx \sqrt{N}\left(1 - \frac{1}{4N} + \frac{1}{21N^2}\right)$。

In [ ]:
def update_step_size(sigma, ps, y_weighted, C, B, D_diag, c_sigma, d_sigma, chi_N, mu_eff):
    '''
    更新步长 sigma 和共轭演化路径 p_sigma
    y_weighted: 加权平均的 (m_new - m_old) / sigma
    '''
    # C^{-1/2} y_weighted = B D^{-1} B^T y_weighted
    inv_sqrt_C_y = B @ np.diag(1.0 / D_diag) @ B.T @ y_weighted

    # 更新共轭演化路径
    ps = (1 - c_sigma) * ps + np.sqrt(c_sigma * (2 - c_sigma) * mu_eff) * inv_sqrt_C_y

    # 更新步长
    sigma = sigma * np.exp(c_sigma / d_sigma * (np.linalg.norm(ps) / chi_N - 1))

    return sigma, ps

### CMA-ES 主循环

整合采样、选择、协方差更新和步长更新。为降低计算开销，每 $N/10$ 代对协方差矩阵做一次特征分解（实际应用中也可每代都做）。

In [ ]:
# ==================== 初始化 ====================
np.random.seed(42)

# 初始化均值（在搜索范围内随机）
m = np.random.uniform(x_min, x_max, D)

# 初始化协方差矩阵为单位矩阵
C = np.eye(D)
B, D_diag = eigen_decompose(C)

# 初始化演化路径
pc = np.zeros(D)
ps = np.zeros(D)

# 初始化步长
sigma = sigma0

# 记录迭代过程
best_fitness_history = []

# 计算初始最优
B, D_diag = eigen_decompose(C)
init_pop = sample_population(m, sigma, B, D_diag, lam)
init_fitness = rastrigin(init_pop)
best_fitness_history.append(np.min(init_fitness))

print(f'初始均值: {np.round(m, 3)}')
print(f'初始最优适应度: {best_fitness_history[0]:.4f}')
print()

# ==================== 主循环 ====================
for gen in range(max_gen):
    # 1. 特征分解（每 D//10 代执行一次，降低计算开销）
    if gen % max(1, D // 10) == 0:
        B, D_diag = eigen_decompose(C)

    # 2. 采样
    pop = sample_population(m, sigma, B, D_diag, lam)

    # 3. 评估适应度
    fitness = rastrigin(pop)

    # 4. 选择与重组
    m_new, selected = select_and_recombine(pop, fitness, weights, mu)

    # 5. 加权平均的 y
    y_weighted = (m_new - m) / sigma

    # 6. 更新步长 sigma
    sigma, ps = update_step_size(sigma, ps, y_weighted, C, B, D_diag,
                                  c_sigma, d_sigma, chi_N, mu_eff)

    # 7. 更新协方差矩阵
    C, pc = update_covariance(C, pc, y_weighted, selected, m, m_new, sigma,
                               weights, c_1, c_mu, c_c, mu_eff)

    # 8. 更新均值
    m = m_new

    # 9. 记录最优
    best_fitness_history.append(np.min(fitness))

    # 打印进度
    if (gen + 1) % 50 == 0 or gen == 0:
        print(f'Generation {gen + 1:4d}, Best Fitness: {best_fitness_history[-1]:.8f}, '
              f'sigma: {sigma:.4f}')

print()
print(f'优化完成！')
print(f'最终最优适应度: {best_fitness_history[-1]:.10f}')
print(f'最优解: {np.round(m, 4)}')
print(f'最终步长 sigma: {sigma:.6f}')

### 收敛曲线

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(best_fitness_history, linewidth=2, color='#2c7bb6')
plt.xlabel('迭代次数', fontsize=12)
plt.ylabel('最优适应度', fontsize=12)
plt.title('CMA-ES 算法收敛曲线 (Rastrigin 函数, D=10)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

### 二维搜索过程可视化

为了直观展示 CMA-ES 的**分布学习特性**，在二维 Rastrigin 函数上运行算法，绘制每一代的采样点及协方差椭圆。

注意观察协方差椭圆如何从初始的圆形逐步变形和收缩，自适应地贴合目标函数的几何结构——这是 CMA-ES 区别于 DE、PSO 等算法的关键特征。

In [ ]:
def plot_covariance_ellipse(ax, mean, cov, sigma_val, color='white', alpha=0.8, n_std=2.0):
    '''
    绘制协方差椭圆（放大到 sigma * n_std 倍）
    '''
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    width = 2 * n_std * sigma_val * np.sqrt(eigenvalues[0])
    height = 2 * n_std * sigma_val * np.sqrt(eigenvalues[1])

    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle,
                      facecolor='none', edgecolor=color, linewidth=1.5, alpha=alpha)
    ax.add_patch(ellipse)


# ==================== 二维 CMA-ES ====================
np.random.seed(42)

D_2d = 2
lam_2d = 4 + int(np.floor(3 * np.log(D_2d)))
mu_2d = lam_2d // 2
weights_2d = np.log((lam_2d + 1) / 2) - np.log(np.arange(1, mu_2d + 1))
weights_2d = weights_2d / np.sum(weights_2d)
mu_eff_2d = 1.0 / np.sum(weights_2d**2)

c_sigma_2d = (mu_eff_2d + 2) / (D_2d + mu_eff_2d + 5)
d_sigma_2d = 1 + 2 * max(0, np.sqrt((mu_eff_2d - 1) / (D_2d + 1)) - 1) + c_sigma_2d
c_c_2d = (4 + mu_eff_2d / D_2d) / (D_2d + 4 + 2 * mu_eff_2d / D_2d)
c_1_2d = 2 / ((D_2d + 1.3)**2 + mu_eff_2d)
c_mu_2d = min(1 - c_1_2d, 2 * (mu_eff_2d - 2 + 1 / mu_eff_2d) / ((D_2d + 2)**2 + mu_eff_2d))
chi_N_2d = np.sqrt(D_2d) * (1 - 1 / (4 * D_2d) + 1 / (21 * D_2d**2))

m_2d = np.random.uniform(x_min, x_max, D_2d)
sigma_2d = 0.5
C_2d = np.eye(D_2d)
pc_2d = np.zeros(D_2d)
ps_2d = np.zeros(D_2d)
max_gen_2d = 200

# 等高线背景
X_bg, Y_bg = np.meshgrid(np.linspace(x_min, x_max, 200),
                           np.linspace(x_min, x_max, 200))
Z_bg = rastrigin(np.column_stack([X_bg.ravel(), Y_bg.ravel()])).reshape(X_bg.shape)

fig, axes = plt.subplots(2, 3, figsize=(16, 11))
axes = axes.flatten()

snapshots = [0, 1, 3, 10, 50, max_gen_2d]

for ax_idx, snap in enumerate(snapshots):
    if snap > 0:
        steps = snap if ax_idx == 0 else snap - snapshots[ax_idx - 1]
        for _ in range(steps):
            B_2d, D_diag_2d = eigen_decompose(C_2d)
            pop_2d = sample_population(m_2d, sigma_2d, B_2d, D_diag_2d, lam_2d)
            fitness_2d = rastrigin(pop_2d)

            m_new_2d, selected_2d = select_and_recombine(pop_2d, fitness_2d,
                                                          weights_2d, mu_2d)
            y_w_2d = (m_new_2d - m_2d) / sigma_2d

            sigma_2d, ps_2d = update_step_size(sigma_2d, ps_2d, y_w_2d, C_2d,
                                                B_2d, D_diag_2d, c_sigma_2d,
                                                d_sigma_2d, chi_N_2d, mu_eff_2d)
            C_2d, pc_2d = update_covariance(C_2d, pc_2d, y_w_2d, selected_2d,
                                             m_2d, m_new_2d, sigma_2d, weights_2d,
                                             c_1_2d, c_mu_2d, c_c_2d, mu_eff_2d)
            m_2d = m_new_2d

    ax = axes[ax_idx]
    ax.contourf(X_bg, Y_bg, Z_bg, levels=30, cmap='viridis', alpha=0.6)

    # 获取当前状态的采样点
    B_2d, D_diag_2d = eigen_decompose(C_2d)
    pop_2d = sample_population(m_2d, sigma_2d, B_2d, D_diag_2d, lam_2d)
    fitness_2d = rastrigin(pop_2d)

    # 绘制采样点
    ax.scatter(pop_2d[:, 0], pop_2d[:, 1], c='red', s=12, alpha=0.6,
               edgecolors='white', linewidth=0.3)
    # 高亮最优个体
    best_idx_2d = np.argmin(fitness_2d)
    ax.scatter(pop_2d[best_idx_2d, 0], pop_2d[best_idx_2d, 1],
               c='yellow', s=80, marker='*', edgecolors='black', linewidth=1)
    # 绘制均值和协方差椭圆
    ax.scatter(m_2d[0], m_2d[1], c='cyan', s=50, marker='D',
               edgecolors='black', linewidth=1, zorder=5)
    plot_covariance_ellipse(ax, m_2d, C_2d, sigma_2d, color='white', alpha=0.9)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(x_min, x_max)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'第 {snap} 代 (sigma={sigma_2d:.3f})')

plt.suptitle('CMA-ES 搜索过程 -- 协方差矩阵自适应学习 (Rastrigin 函数)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 一点讨论

#### CMA-ES 的关键优势

**1. 不变性（Invariance）**

CMA-ES 是少数具有完全不变性的进化算法之一：

- **平移不变性**：均值更新 $\mathbf{m}$ 使算法对目标函数的平移不敏感
- **旋转不变性**：协方差矩阵 $\mathbf{C}$ 能自适应旋转，使算法不依赖坐标系的选择
- **尺度不变性**：步长 $\sigma$ 独立自适应，使算法对不同尺度的变量同样有效

这意味着在任意线性变换后的搜索空间中，CMA-ES 的行为保持一致——这是 DE 和 PSO 无法做到的。

**2. 参数自适应**

- 只需设置初始步长 $\sigma_0$ 和初始均值 $\mathbf{m}_0$，其余参数全部自动推导
- 协方差矩阵自动学习变量间的相关性和各维度的敏感度
- 步长 $\sigma$ 根据搜索状态自动调整，无需人工干预

**3. 适用场景**

- **高维优化**（20~100 维）：CMA-ES 表现优于大多数进化算法
- **ill-conditioned 问题**：协方差矩阵能近似捕获 Hessian 的逆信息
- **非凸/多模态问题**：足够大的种群规模保证全局搜索能力

#### 参数选择建议

| 参数 | 建议值 | 说明 |
|------|--------|------|
| $\sigma_0$ | $0.25 \times$ 搜索范围 | 初始步长，影响早期探索范围 |
| $\lambda$ | $4 + 3\ln D$ | 默认值通常足够；增大可提升全局搜索能力 |
| $\mu$ | $\lfloor \lambda/2 \rfloor$ | 父代数量；增大使搜索更保守 |

#### CMA-ES vs DE vs PSO

| 特性 | CMA-ES | DE | PSO |
|------|--------|-----|-----|
| 自适应能力 | 协方差矩阵 + 步长全自适应 | 无（参数固定） | 仅惯性权重可调 |
| 用户需设参数 | 极少（仅 $\sigma_0$） | 3 个（$NP$, $F$, $CR$） | 较多（$w$, $c_1$, $c_2$, $pop$） |
| 高维性能 | ★★★★★ | ★★★ | ★★ |
| 实现复杂度 | 高 | 低 | 低 |
| 理论完备性 | 有完整理论分析 | 启发式 | 启发式 |
| 不变性 | 平移/旋转/尺度 | 旋转 | 无 |